In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install fastapi uvicorn python-multipart pyngrok rapidfuzz librosa transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os, re, json, tempfile, threading
import librosa, torch, rapidfuzz
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import uvicorn
from pyngrok import ngrok

# ============================================================
# CONFIG
# ============================================================
ORIGINAL_MODEL = "openai/whisper-small"
MY_CHECKPOINT  = "/content/drive/MyDrive/whisper-quran-warsh-v2/checkpoint-200"
QURAN_JSON_DIR = "/content/drive/MyDrive/warsh_json"
NGROK_TOKEN    = "365frfowFUSZULaZbTFl5C7y63J_77k3W5vTfZKtCWHnpfrEW"

# ============================================================
# DEVICE
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device: {device}")

# ============================================================
# LOAD MODEL
# ============================================================
print("⏳ Loading model...")
processor = WhisperProcessor.from_pretrained(ORIGINAL_MODEL, language="arabic", task="transcribe")
model     = WhisperForConditionalGeneration.from_pretrained(MY_CHECKPOINT).to(device)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="arabic", task="transcribe")
print("✅ Model loaded!")

# ============================================================
# LOAD QURAN JSON
# ============================================================
print("⏳ Loading Quran JSON...")
QURAN_DATA = {}
for fname in os.listdir(QURAN_JSON_DIR):
    if fname.endswith(".json"):
        with open(os.path.join(QURAN_JSON_DIR, fname), "r", encoding="utf-8") as f:
            data = json.load(f)
        QURAN_DATA[data["surah_number"]] = data
print(f"✅ Loaded {len(QURAN_DATA)} surahs")

# ============================================================
# NORMALIZE — خاصة برواية ورش
# ============================================================
def normalize(text: str) -> str:
    # 1. حذف الحركات العادية
    text = re.sub(r'[\u064B-\u065F]', '', text)
    # 2. حذف علامات التجويد
    text = re.sub(r'[\u06D6-\u06DC\u06DF-\u06ED]', '', text)
    # 3. الألف الصغيرة ٰ (U+0670) → ا
    text = text.replace('\u0670', 'ا')
    # 4. أإآٱ → ا
    text = re.sub(r'[أإآٱ]', 'ا', text)
    # 5. مسافات زائدة
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ============================================================
# إزالة البسملة من أول الكلام
# ============================================================
def remove_basmala(words: list) -> list:
    if len(words) >= 4:
        first_four = normalize(" ".join(words[:4]))
        basmala    = normalize("بسم الله الرحمن الرحيم")
        if rapidfuzz.fuzz.ratio(basmala, first_four) >= 70:
            return words[4:]
    if len(words) >= 3:
        first_three = normalize(" ".join(words[:3]))
        basmala3    = normalize("الله الرحمن الرحيم")
        if rapidfuzz.fuzz.ratio(basmala3, first_three) >= 70:
            return words[3:]
    return words

# ============================================================
# ✅ الإصلاح الجديد — فصل الكلمات الملتصقة
# ============================================================
def expand_merged_words(spoken_words: list, expected_texts: list, max_merge: int = 3) -> list:
    """
    Whisper أحياناً يلصق كلمتين أو أكثر بدون مسافة.
    مثلاً: "العالمينالرحمان" بدل "العالمين الرحمان"

    الخوارزمية:
    - لكل كلمة منطوقة، نجرب مطابقتها مع 1 أو 2 أو 3 كلمات متوقعة ملتصقة
    - نختار الاحتمال الأعلى score
    - إذا كانت ملتصقة، نفصلها ونرجع الكلمات المتوقعة المقابلة
    """
    result  = []
    exp_idx = 0

    for word in spoken_words:
        if exp_idx >= len(expected_texts):
            result.append(word)
            continue

        best_score = 0
        best_n     = 1

        for n in range(1, max_merge + 1):
            if exp_idx + n > len(expected_texts):
                break
            merged = "".join(expected_texts[exp_idx : exp_idx + n])
            score  = rapidfuzz.fuzz.ratio(word, merged)
            if score > best_score:
                best_score = score
                best_n     = n

        if best_score >= 65:
            # أضيف الكلمات المتوقعة المفصولة
            for k in range(best_n):
                result.append(expected_texts[exp_idx + k])
            exp_idx += best_n
        else:
            result.append(word)
            exp_idx += 1

    return result

# ============================================================
# TRANSCRIBE
# ============================================================
def transcribe(audio_path: str) -> str:
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs   = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    with torch.no_grad():
        ids = model.generate(inputs)
    return processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0].strip()

# ============================================================
# CORRECTION — منطق الأخطاء (مع الإصلاح)
# ============================================================
def correct_words(spoken_text: str, expected_words: list) -> dict:
    spoken_normalized = normalize(spoken_text)
    spoken_words_raw  = spoken_normalized.split()

    # إزالة البسملة إذا قالها
    spoken_words_raw = remove_basmala(spoken_words_raw)

    # ✅ فصل الكلمات الملتصقة قبل المقارنة
    expected_texts = [w["text_normalized"] for w in expected_words]
    spoken_words   = expand_merged_words(spoken_words_raw, expected_texts, max_merge=3)

    results            = []
    consecutive_errors = 0
    first_error_index  = None
    stopped            = False
    stop_at            = None

    for i, spoken_word in enumerate(spoken_words):
        if i >= len(expected_words):
            break

        expected_norm = expected_words[i]["text_normalized"]
        score         = rapidfuzz.fuzz.partial_ratio(expected_norm, spoken_word)
        is_correct    = score >= 75

        results.append({
            "word_index"    : expected_words[i]["word_index"],
            "text_original" : expected_words[i]["text_original"],
            "state"         : "correct" if is_correct else "wrong",
            "score"         : score,
            "spoken_word"   : spoken_word
        })

        if is_correct:
            consecutive_errors = 0
            first_error_index  = None
        else:
            if consecutive_errors == 0:
                first_error_index = i
            consecutive_errors += 1

            if consecutive_errors >= 3:
                stop_at = first_error_index
                stopped = True
                results = results[:first_error_index + 1]
                break

    return {
        "spoken_text"       : spoken_text,
        "spoken_normalized" : spoken_normalized,
        "results"           : results,
        "stopped"           : stopped,
        "stop_at_word_index": stop_at,
        "stop_message"      : (
            f"ابدأ من كلمة: {expected_words[stop_at]['text_original']}"
            if stop_at is not None else None
        ),
        "consecutive_errors": consecutive_errors
    }

# ============================================================
# HELPER
# ============================================================
def get_all_words(surah_number: int) -> list:
    all_words = []
    for aya in QURAN_DATA[surah_number]["ayat"]:
        for w in aya["words"]:
            all_words.append({
                "word_index"     : len(all_words),
                "aya_number"     : aya["aya_number"],
                "text_original"  : w["text_original"],
                "text_normalized": w["text_normalized"]
            })
    return all_words

# ============================================================
# FASTAPI
# ============================================================
app = FastAPI(title="Quran Correction API — Warsh", version="3.1.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

@app.get("/health")
async def health():
    return {"status": "ok", "device": device, "surahs_loaded": len(QURAN_DATA)}

@app.get("/surahs")
async def list_surahs():
    return [
        {
            "surah_number" : v["surah_number"],
            "surah_name"   : v["surah_name"],
            "surah_name_en": v["surah_name_en"],
            "total_ayat"   : v["total_ayat"]
        }
        for v in sorted(QURAN_DATA.values(), key=lambda x: x["surah_number"])
    ]

@app.get("/surah/{surah_number}")
async def get_surah(surah_number: int):
    if surah_number not in QURAN_DATA:
        raise HTTPException(status_code=404, detail="Surah not found")
    return QURAN_DATA[surah_number]

@app.get("/surah/{surah_number}/aya/{aya_number}")
async def get_aya(surah_number: int, aya_number: int):
    if surah_number not in QURAN_DATA:
        raise HTTPException(status_code=404, detail="Surah not found")
    ayat = QURAN_DATA[surah_number]["ayat"]
    if aya_number < 1 or aya_number > len(ayat):
        raise HTTPException(status_code=404, detail="Aya not found")
    return ayat[aya_number - 1]

@app.post("/transcribe")
async def transcribe_audio(audio: UploadFile = File(...)):
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp.write(await audio.read())
        tmp_path = tmp.name
    try:
        text = transcribe(tmp_path)
        return {"text": text, "normalized": normalize(text)}
    finally:
        os.unlink(tmp_path)

@app.post("/recite")
async def recite(
    audio         : UploadFile = File(...),
    surah_number  : int        = Form(...),
    start_word_abs: int        = Form(0),
    word_count    : int        = Form(100),
):
    if surah_number not in QURAN_DATA:
        raise HTTPException(status_code=404, detail="Surah not found")

    all_words      = get_all_words(surah_number)
    end            = min(start_word_abs + word_count, len(all_words))
    expected_words = all_words[start_word_abs:end]

    if not expected_words:
        return {"message": "🎉 أكملت السورة!", "finished": True}

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp.write(await audio.read())
        tmp_path = tmp.name

    try:
        spoken_text = transcribe(tmp_path)
        correction  = correct_words(spoken_text, expected_words)

        if correction["stopped"]:
            next_word_abs = start_word_abs + correction["stop_at_word_index"]
        else:
            next_word_abs = start_word_abs + len(correction["results"])

        correction["next_word_abs"] = next_word_abs
        correction["finished"]      = (next_word_abs >= len(all_words))
        correction["total_words"]   = len(all_words)
        correction["progress"]      = round(next_word_abs / len(all_words) * 100, 1)

        return correction

    finally:
        os.unlink(tmp_path)

# ============================================================
# LAUNCH
# ============================================================
os.system("fuser -k 8000/tcp 2>/dev/null")
import time; time.sleep(1)

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(
    8000,
    domain="acapnial-macrostylous-tyra.ngrok-free.dev"
)
print(f"\n✅ API is live!")
print(f"🌐 URL  : {public_url.public_url}")
print(f"📖 Docs : {public_url.public_url}/docs\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
server.install_signal_handlers = lambda: None

thread = threading.Thread(target=server.run)
thread.daemon = True
thread.start()
print("✅ Server running!")

🖥️ Device: cpu
⏳ Loading model...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

✅ Model loaded!
⏳ Loading Quran JSON...
✅ Loaded 29 surahs

✅ API is live!
🌐 URL  : https://acapnial-macrostylous-tyra.ngrok-free.dev
📖 Docs : https://acapnial-macrostylous-tyra.ngrok-free.dev/docs

✅ Server running!
